<a href="https://colab.research.google.com/github/moise97/Extract_-_Structure_Data_from_SDFs_pharmaceutical_documentation/blob/main/Design_a_Page_Level_Detection_Strategy_Using_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q google-generativeai
!pip install -q pypdf
!pip install -q pandas
!pip install -q Pillow
!pip install -q pdf2image
!apt-get install -q poppler-utils  # needed by pdf2image

print('✅ All packages installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 6.8 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (1,039 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118252 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
✅ All packages installed!


In [6]:
from google.colab import files

print('pharma-blob-sample.pdf')
uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]
print(f'\n✅ Uploaded: {pdf_filename}')

pharma-blob-sample.pdf


Saving pharma-blob-sample.pdf to pharma-blob-sample (2).pdf

✅ Uploaded: pharma-blob-sample (2).pdf


In [7]:
import google.generativeai as genai

# ✏️ PASTE YOUR GEMINI API KEY HERE
GEMINI_API_KEY = 'YOUR_GEMINI_API_KEY_HERE'

# --------------------------------------------------------

if GEMINI_API_KEY == 'YOUR_GEMINI_API_KEY_HERE':
    print('❌ Please paste your Gemini API key above!')
else:
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel('gemini-1.5-flash')
    print('✅ Gemini configured — using gemini-1.5-flash')

❌ Please paste your Gemini API key above!


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [8]:
from pdf2image import convert_from_path
from PIL import Image
import os

print(f'Converting PDF pages to images...')
page_images = convert_from_path(pdf_filename, dpi=150)

total_pages = len(page_images)
print(f'✅ Converted {total_pages} page(s) to images')

# Save pages to disk so we can reference them
os.makedirs('pages', exist_ok=True)
for i, img in enumerate(page_images):
    img.save(f'pages/page_{i}.png', 'PNG')

print(f'   Pages saved to /pages/page_0.png ... page_{total_pages-1}.png')

Converting PDF pages to images...
✅ Converted 10 page(s) to images
   Pages saved to /pages/page_0.png ... page_9.png


In [9]:
import time

def is_same_document(page_img_a: Image.Image, page_img_b: Image.Image) -> bool:
    """
    Returns True if both pages belong to the same document.
    Returns False if page_b is the start of a new document.
    """
    prompt = """
You are a document analysis assistant.

You are given TWO pages from a scanned PDF that may contain multiple separate documents.

Your job: decide if these two pages belong to the SAME document, or if the second page is the START of a NEW document.

Look for these signals that it's a NEW document:
- Different document title or header
- Different company name or logo
- Different document type (e.g. invoice vs certificate)
- A clear "Page 1" or fresh header on the second page
- Change in formatting, font, or layout style

Look for these signals it's the SAME document:
- Continues from the previous page (e.g. "Page 2 of 3")
- Same header, footer, or document number
- Text that clearly continues mid-sentence or mid-table

Respond with ONLY one word: YES (same document) or NO (new document).
"""
    try:
        response = model.generate_content([prompt, page_img_a, page_img_b])
        answer = response.text.strip().upper()
        return answer.startswith('YES')
    except Exception as e:
        print(f'  ⚠️  is_same_document error: {e} — defaulting to same document')
        return True


def classify_document_type(page_img: Image.Image) -> str:
    """
    Returns the document type label for the given page image.
    """
    prompt = """
You are a document classification assistant.

Look at this page and identify the type of document it belongs to.

Common document types include (but are not limited to):
- Cover Letter
- Certificate of Quality
- Certificate of Conformance
- Packing List
- Commercial Invoice
- Purchase Order
- Bill of Lading
- Material Safety Data Sheet (MSDS)
- Test Report
- Delivery Note
- Technical Specification
- Declaration of Conformity
- Other

Respond with ONLY the document type name — nothing else. No explanation, no punctuation.
"""
    try:
        response = model.generate_content([prompt, page_img])
        return response.text.strip()
    except Exception as e:
        print(f'  ⚠️  classify_document_type error: {e} — labelling as Unknown')
        return 'Unknown'


print('✅ Functions defined:')
print('   is_same_document(page_a, page_b)  → True / False')
print('   classify_document_type(page)      → document type string')

✅ Functions defined:
   is_same_document(page_a, page_b)  → True / False
   classify_document_type(page)      → document type string


In [10]:
results = []
doc_id = 0
current_doc_type = None
DELAY = 1.5  # seconds between API calls to avoid rate limits

print(f'Starting classification loop over {total_pages} pages...\n')
print('-' * 55)

for page_num in range(total_pages):
    current_page = page_images[page_num]

    if page_num == 0:
        # First page — always a new document
        print(f'Page {page_num}: First page → classifying...')
        current_doc_type = classify_document_type(current_page)
        print(f'           doc_id={doc_id} | type="{current_doc_type}"')

    else:
        prev_page = page_images[page_num - 1]

        # Ask Gemini: same doc or new doc?
        same = is_same_document(prev_page, current_page)

        if same:
            print(f'Page {page_num}: Same document → doc_id={doc_id} | "{current_doc_type}"')
        else:
            doc_id += 1
            print(f'Page {page_num}: NEW document detected → classifying...')
            current_doc_type = classify_document_type(current_page)
            print(f'           doc_id={doc_id} | type="{current_doc_type}"')

    results.append({
        'page':     page_num,
        'doc_id':   doc_id,
        'doc_type': current_doc_type
    })

    time.sleep(DELAY)  # be polite to the API

print('-' * 55)
print(f'\n✅ Done! Processed {total_pages} pages → found {doc_id + 1} document(s).')

Starting classification loop over 10 pages...

-------------------------------------------------------
Page 0: First page → classifying...
  ⚠️  classify_document_type error: name 'model' is not defined — labelling as Unknown
           doc_id=0 | type="Unknown"
  ⚠️  is_same_document error: name 'model' is not defined — defaulting to same document
Page 1: Same document → doc_id=0 | "Unknown"
  ⚠️  is_same_document error: name 'model' is not defined — defaulting to same document
Page 2: Same document → doc_id=0 | "Unknown"
  ⚠️  is_same_document error: name 'model' is not defined — defaulting to same document
Page 3: Same document → doc_id=0 | "Unknown"
  ⚠️  is_same_document error: name 'model' is not defined — defaulting to same document
Page 4: Same document → doc_id=0 | "Unknown"
  ⚠️  is_same_document error: name 'model' is not defined — defaulting to same document
Page 5: Same document → doc_id=0 | "Unknown"
  ⚠️  is_same_document error: name 'model' is not defined — defaulting t

In [11]:
import pandas as pd

df = pd.DataFrame(results)

print('Classification Results:')
print('=' * 45)
print(df.to_string(index=False))
print('=' * 45)
print(f'Total pages   : {len(df)}')
print(f'Total documents: {df["doc_id"].nunique()}')
print(f'Document types : {df.groupby("doc_id")["doc_type"].first().tolist()}')

Classification Results:
 page  doc_id doc_type
    0       0  Unknown
    1       0  Unknown
    2       0  Unknown
    3       0  Unknown
    4       0  Unknown
    5       0  Unknown
    6       0  Unknown
    7       0  Unknown
    8       0  Unknown
    9       0  Unknown
Total pages   : 10
Total documents: 1
Document types : ['Unknown']


In [12]:
import json

# Pretty-print JSON to console
json_output = json.dumps(results, indent=2)
print('JSON Output (copy this into your Google Doc):')
print('=' * 55)
print(json_output)
print('=' * 55)

# Save to file and download
with open('classification_results.json', 'w') as f:
    f.write(json_output)

print('\n✅ Saved as classification_results.json')

from google.colab import files
files.download('classification_results.json')

JSON Output (copy this into your Google Doc):
[
  {
    "page": 0,
    "doc_id": 0,
    "doc_type": "Unknown"
  },
  {
    "page": 1,
    "doc_id": 0,
    "doc_type": "Unknown"
  },
  {
    "page": 2,
    "doc_id": 0,
    "doc_type": "Unknown"
  },
  {
    "page": 3,
    "doc_id": 0,
    "doc_type": "Unknown"
  },
  {
    "page": 4,
    "doc_id": 0,
    "doc_type": "Unknown"
  },
  {
    "page": 5,
    "doc_id": 0,
    "doc_type": "Unknown"
  },
  {
    "page": 6,
    "doc_id": 0,
    "doc_type": "Unknown"
  },
  {
    "page": 7,
    "doc_id": 0,
    "doc_type": "Unknown"
  },
  {
    "page": 8,
    "doc_id": 0,
    "doc_type": "Unknown"
  },
  {
    "page": 9,
    "doc_id": 0,
    "doc_type": "Unknown"
  }
]

✅ Saved as classification_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df.to_csv('classification_results.csv', index=False)
print('✅ Saved as classification_results.csv')

from google.colab import files
files.download('classification_results.csv')